# Top Changed Neurons: Normal → Anhedonic
Identifies the exact (layer, neuron) pairs most affected by the anhedonic condition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size']  = 11

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
neu_normal    = np.load('activations/activations_neurons_normal.npy')
neu_anhedonic = np.load('activations/activations_neurons_anhedonic.npy')
neu_neutral   = np.load('activations/activations_neurons_neutral.npy')

n_prompts, n_layers, n_neurons = neu_normal.shape
print(f'Shape: {neu_normal.shape}  →  (prompts={n_prompts}, layers={n_layers}, neurons={n_neurons})')

In [ ]:
# ── Compute per-neuron statistics ─────────────────────────────────────────────
mean_normal    = neu_normal.mean(axis=0)       # (28, 18944)
mean_anhedonic = neu_anhedonic.mean(axis=0)
mean_neutral   = neu_neutral.mean(axis=0)

diff_signed = mean_anhedonic - mean_normal
diff_abs    = np.abs(diff_signed)

# Effect size (Cohen's d)
std_normal    = neu_normal.std(axis=0)
std_anhedonic = neu_anhedonic.std(axis=0)
pooled_std    = np.sqrt((std_normal**2 + std_anhedonic**2) / 2 + 1e-8)
effect_size   = diff_signed / pooled_std

print(f'Global max |Δ|        : {diff_abs.max():.4f}')
print(f'Global max effect size: {np.abs(effect_size).max():.4f}')

In [ ]:
# ── Extract top 50 neurons globally ──────────────────────────────────────────
TOP_N    = 50
flat_idx = np.argsort(diff_abs.flatten())[::-1][:TOP_N]
top_l    = flat_idx // n_neurons
top_n    = flat_idx %  n_neurons

df_top = pd.DataFrame({
    'rank':           range(1, TOP_N + 1),
    'layer':          top_l,
    'neuron':         top_n,
    'abs_delta':      diff_abs.flatten()[flat_idx],
    'signed_delta':   diff_signed[top_l, top_n],
    'effect_size':    effect_size[top_l, top_n],
    'direction':      ['UP' if d > 0 else 'DOWN' for d in diff_signed[top_l, top_n]],
    'mean_normal':    mean_normal[top_l, top_n],
    'mean_anhedonic': mean_anhedonic[top_l, top_n],
})

print(df_top[['rank','layer','neuron','signed_delta','effect_size','direction']].to_string(index=False))

## Plot 1: Top 30 Neurons Ranked by |Δ|

In [ ]:
TOP_PLOT = 30
df_plot  = df_top.head(TOP_PLOT)

fig, ax = plt.subplots(figsize=(14, 8))
colors  = ['#D32F2F' if d > 0 else '#1565C0' for d in df_plot['signed_delta']]
labels  = [f'L{int(l):02d} · N{int(n):05d}' for l, n in zip(df_plot['layer'], df_plot['neuron'])]

ax.barh(range(TOP_PLOT), df_plot['signed_delta'], color=colors, alpha=0.85, height=0.7)
ax.set_yticks(range(TOP_PLOT))
ax.set_yticklabels(labels, fontsize=9, fontfamily='monospace')
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ Mean Activation (Anhedonic − Normal)', fontsize=12)
ax.set_title('Top 30 Most Changed Neurons: Normal → Anhedonic\n'
             'Red = increased  |  Blue = decreased  |  (d = Cohen effect size)', fontsize=13)
ax.grid(axis='x', alpha=0.3)

for i, (_, row) in enumerate(df_plot.iterrows()):
    x   = row['signed_delta']
    ha  = 'left' if x >= 0 else 'right'
    off = 0.001 if x >= 0 else -0.001
    ax.text(x + off, i, f" d={row['effect_size']:.2f}", va='center', ha=ha, fontsize=7.5)

plt.tight_layout()
plt.savefig('fig_top30_neurons.png', bbox_inches='tight')
plt.show()

## Plot 2: Layer Distribution of Top 50 Neurons

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total count per layer
ax = axes[0]
layer_counts = df_top['layer'].value_counts().reindex(range(n_layers), fill_value=0).sort_index()
ax.bar(range(n_layers), layer_counts.values, color=plt.cm.plasma(layer_counts.values / layer_counts.max() + 0.1), alpha=0.9)
ax.set_xlabel('Layer')
ax.set_ylabel('# Neurons in Top 50')
ax.set_title('Which Layers Contain the Most Changed Neurons?')
ax.set_xticks(range(n_layers))
ax.grid(axis='y', alpha=0.3)

# Up vs down per layer
ax = axes[1]
up_df   = df_top[df_top['signed_delta'] > 0]
down_df = df_top[df_top['signed_delta'] < 0]
up_vals   = up_df['layer'].value_counts().reindex(range(n_layers), fill_value=0).sort_index().values
down_vals = down_df['layer'].value_counts().reindex(range(n_layers), fill_value=0).sort_index().values

ax.bar(range(n_layers),  up_vals,   label='Increased (anhedonic)', color='#D32F2F', alpha=0.8)
ax.bar(range(n_layers), -down_vals, label='Decreased (anhedonic)', color='#1565C0', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Count (+ up / - down)')
ax.set_title('Direction of Change per Layer')
ax.set_xticks(range(n_layers))
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_top_neurons_by_layer.png', bbox_inches='tight')
plt.show()

## Plot 3: Activation Distributions — Top 12 Neurons (Violin)

In [ ]:
TOP_DETAIL = 12
df_detail  = df_top.head(TOP_DETAIL)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, (_, row) in enumerate(df_detail.iterrows()):
    l, n = int(row['layer']), int(row['neuron'])
    ax   = axes[i]

    vals = [neu_normal[:, l, n], neu_anhedonic[:, l, n], neu_neutral[:, l, n]]
    parts = ax.violinplot(vals, positions=[0,1,2], showmeans=True, showmedians=False)

    for pc, color in zip(parts['bodies'], ['#2196F3','#F44336','#4CAF50']):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Normal', 'Anhedonic', 'Neutral'], fontsize=8)
    ax.set_title(f'Layer {l}  ·  Neuron {n}\nΔ={row["signed_delta"]:+.3f}   d={row["effect_size"]:+.2f}', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    for pos, v, c in [(0, vals[0], '#2196F3'), (1, vals[1], '#F44336'), (2, vals[2], '#4CAF50')]:
        ax.text(pos, ax.get_ylim()[1]*0.97, f'{v.mean():.3f}', ha='center', fontsize=7, color=c, fontweight='bold')

plt.suptitle('Activation Distribution: Top 12 Most Changed Neurons\n'
             'Blue = Normal   Red = Anhedonic   Green = Neutral', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_top12_violin.png', bbox_inches='tight')
plt.show()

## Plot 4: Heatmap — Top 100 Neurons per Layer

In [ ]:
TOP_PER_LAYER = 100
heatmap_data  = np.zeros((n_layers, TOP_PER_LAYER))
for l in range(n_layers):
    idx = np.argsort(diff_abs[l])[::-1][:TOP_PER_LAYER]
    heatmap_data[l] = diff_signed[l, idx]

fig, ax = plt.subplots(figsize=(14, 7))
vmax = np.percentile(np.abs(heatmap_data), 98)
im   = ax.imshow(heatmap_data, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xlabel(f'Neuron rank within layer (top {TOP_PER_LAYER} by |Δ|)')
ax.set_ylabel('Layer')
ax.set_yticks(range(n_layers))
ax.set_yticklabels([f'L{i:02d}' for i in range(n_layers)], fontsize=8)
ax.set_title('Signed Δ Activation (Anhedonic − Normal)\n'
             f'Top {TOP_PER_LAYER} most changed neurons per layer   |   Red = increased   Blue = decreased', fontsize=12)
plt.colorbar(im, ax=ax, label='Δ Mean Activation')
plt.tight_layout()
plt.savefig('fig_heatmap_top_per_layer.png', bbox_inches='tight')
plt.show()

## Final Table + Save

In [ ]:
print('='*65)
print('TOP 50 MOST CHANGED NEURONS — Exact Layer & Neuron IDs')
print('='*65)
print(f'{"Rank":>4} {"Layer":>6} {"Neuron":>8} {"Δ Mean":>10} {"Effect d":>10} {"Dir":>6}')
print('-'*65)
for _, row in df_top.iterrows():
    print(f"{int(row['rank']):>4} {int(row['layer']):>6} {int(row['neuron']):>8} "
          f"{row['signed_delta']:>+10.4f} {row['effect_size']:>+10.4f} {row['direction']:>6}")

df_top.to_csv('top_neurons_anhedonic.csv', index=False)
print('\nSaved → top_neurons_anhedonic.csv')